# Predicción de Popularidad Musical en Spotify
## Notebook 2 (EV2, versión corregida): Modelamiento, Evaluación y Análisis Ético

**Asignatura:** MLY1101 Machine Learning · Duoc UC — **Grupo 8 · Autores:** Johan Soto, FernandaAlcaino
**Caso C:** Spotify Tracks — continuación del notebook de EV1 (EDA y limpieza)

### Resumen
Este notebook implementa la fase de modelamiento sobre `spotify_tracks_clean.csv` (generado en EV1):
síntesis del EDA, preparación de datos, tres modelos supervisados (Regresión Logística, Random Forest, XGBoost),
un modelo no supervisado (PCA + K-Means), evaluación con métricas para clases desbalanceadas, análisis por género,
por segmento sonoro y por tipo de artista, y análisis de sesgos y ética.

### Cambios respecto a la versión evaluada (respuesta a la retroalimentación EP2)
| Observación del docente | Corrección aplicada | Sección |
|---|---|---|
| `album_te` filtra información y no existe al lanzar; el split aleatorio deja el mismo álbum/artista (y la misma canción con otro `track_id`) en train y test | **Split por artista** (`GroupShuffleSplit`, ningún artista en ambos conjuntos) y **se elimina el Target Encoding de álbum y de artista** del modelo final. El modelo usa solo información disponible al lanzar (audio, ingeniería, género) | 7, 8 |
| IE8: la ablación se afirmó pero no se mostró | **Tabla de ablación** con 6 conjuntos de variables, evaluada con split aleatorio y con split por artista para cuantificar la fuga | 8 |
| IE8: "se subestima a artistas emergentes" sin cifra | **Recall por tamaño de catálogo** (1 canción, 2–10, más de 10); la afirmación se reemplaza por lo que muestran los datos | 15 |
| IE8: conclusión sobre clusters demasiado fuerte | Reformulada con cautela y con recall por cluster | 16, 20 |
| IE5: KPIs del inicio distintos a los verificados; MAE nunca calculado | **Una sola tabla de KPIs** (`kpis_spotify.csv`, la misma de EV1) definida al inicio y verificada completa al final; se agrega un modelo de **regresión** para MAE/RMSE/R² | 1, 17, 21 |
| IE5: ruta `spotify_tracks_clean.csv` relativa a la carpeta actual | Función `ubicar()` que busca en `../data/` (estructura del repositorio) | 2 |
| IE5: salida desactualizada en la última celda | Mensajes generados con la misma condición que se evalúa; ejecutar *Restart & Run All* antes de entregar | 21 |
| IE5: nombrar CRISP-DM | Mapa explícito de fases CRISP-DM | 0 |
| Presentación: cómo priorizar artistas que invierten en publicidad | Sección con 4 alternativas técnicas y su riesgo (efecto Mateo) | 19 |
| Selección del "mejor modelo" mirando el test | El mejor modelo se elige con el conjunto de **validación**; el test solo confirma | 11 |

> **Tiempo de ejecución estimado:** 10–15 min en Colab (`MODO_RAPIDO = True` ≈ 5 min).

## 0. Metodología CRISP-DM

| Fase CRISP-DM | Qué se hace en este proyecto | Dónde |
|---|---|---|
| 1. Comprensión del negocio | Problema, objetivos y KPIs | EV1 · Sección 1 |
| 2. Comprensión de los datos | EDA, calidad, sesgos | EV1 · Secciones 3–4 |
| 3. Preparación de los datos | Limpieza (EV1), ingeniería de variables, partición por artista, pipeline | Secciones 5–9 |
| 4. Modelado | Regresión Logística, Random Forest, XGBoost, PCA + K-Means, regresión de popularidad | Secciones 10–11, 16–17 |
| 5. Evaluación | Métricas de clasificación, por género, por artista, por cluster, KPIs | Secciones 12–18, 21 |
| 6. Despliegue | Exportación de predicciones y recomendaciones de uso responsable | Secciones 19, 22 |

## 1. Comprensión del problema de negocio y KPIs

### Contexto
Los sellos deben decidir en qué canciones invertir presupuesto promocional **antes de conocer su desempeño**.
La pregunta de negocio es: ¿es posible anticipar, **solo con información disponible al momento del lanzamiento**,
si una canción alcanzará una popularidad relevante?

### Objetivo general
Estimar la probabilidad de que una canción sea un **hit** (`popularity ≥ 50`, ≈ percentil 75, prevalencia ≈ 23.7%).

### Objetivos específicos
- Comparar tres modelos supervisados de clasificación (Regresión Logística, Random Forest, XGBoost).
- Identificar segmentos sonoros con PCA y K-Means y analizar su relación con los hits.
- Evaluar con métricas adecuadas al desbalance y reportar el desempeño por género y por tipo de artista.
- Cuantificar los sesgos detectados en el EDA de EV1.

### Criterio de "información disponible al lanzar"
Una variable entra al modelo solo si se conoce **antes** de publicar la canción y **no contiene** la popularidad
de otras canciones relacionadas: audio, duración, explícito, género, número de artistas y tamaño del catálogo del
artista. La popularidad media del álbum o del artista **no** cumple el criterio (se calcula con reproducciones
posteriores y, como se verá, filtra información).

### KPIs del proyecto (tabla única: se define aquí y se verifica completa en la Sección 21)

In [ ]:
import os, time, textwrap, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn, xgboost
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (train_test_split, GroupShuffleSplit, StratifiedKFold,
                                     StratifiedGroupKFold)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (f1_score, fbeta_score, recall_score, precision_score, accuracy_score,
                             roc_auc_score, average_precision_score, brier_score_loss,
                             precision_recall_curve, confusion_matrix, classification_report,
                             silhouette_score, ndcg_score, mean_absolute_error, mean_squared_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor

RANDOM_STATE = 42
UMBRAL_HIT   = 50
MODO_RAPIDO  = False
N_ARBOLES_RF = 200 if MODO_RAPIDO else 500

np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 200); pd.set_option('display.width', 200)
sns.set_style('whitegrid')
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False, 'axes.titleweight': 'bold'})
COLORES = {'Regresión Logística': '#eda100', 'Random Forest': '#1baf7a', 'XGBoost': '#2a78d6'}

ROOT = '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'
DIRS = {k: os.path.join(ROOT, k) for k in ['data', 'models', 'images', 'reports']}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

def ubicar(nombre):
    """Busca el archivo en ../data (estructura del repositorio), data/, carpeta actual o Colab."""
    for base in ['../data', 'data', DIRS['data'], '.', '/content', '/content/drive/MyDrive']:
        p = os.path.join(base, nombre)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"No se encontró {nombre}. Colóquelo en la carpeta data/ del proyecto.")

print(f"scikit-learn {sklearn.__version__} | xgboost {xgboost.__version__} | pandas {pd.__version__}")

In [ ]:
kpis = pd.read_csv(ubicar('kpis_spotify.csv'))
medicion = {
    'MAE':             'XGBRegressor sobre popularity, set de lanzamiento (Sección 17)',
    'RMSE':            'XGBRegressor sobre popularity, set de lanzamiento (Sección 17)',
    'R2':              'R² del regresor usando SOLO variables musicales: audio + ingeniería (Sección 17)',
    'Recall@Hit':      'Recall en test al umbral F2 (Sección 12)',
    'PR-AUC':          'Average Precision en test (Sección 12)',
    'F2-Score':        'F-beta, beta=2, en test (Sección 12)',
    'NDCG@20':         'Orden de las 20 primeras canciones por género vs popularidad real (Sección 18)',
    'Waste Rate':      'FP / (FP + TP) en test (Sección 13)',
    'Diversity Score': 'Nº de géneros en el top-200 de canciones a promocionar (Sección 18)',
}
kpis['Cómo se mide'] = kpis['KPI'].map(medicion)
display(kpis[['KPI', 'Tipo de Analisis', 'Meta Sugerida', 'Cómo se mide']])

## 2. Configuración

Las rutas usan `ubicar()`, que prioriza `../data/` (el notebook vive en `notebooks/` dentro del repositorio).
Requisitos: scikit-learn ≥ 1.4, xgboost ≥ 2.1.4.

## 3. Carga del dataset, verificación de calidad e ingeniería de características

Validación defensiva (columnas críticas sin nulos, target bien formado) y las variables derivadas de EV2:
- `duration_min`, `is_short`, `is_long`: el EDA mostró un óptimo de 3–4 minutos y caída en los extremos.
- `n_artists`: colaboraciones y solistas tienen patrones distintos.
- `energy_x_dance`, `valence_x_energy`, `explicit_x_dance`: perfiles típicos de hit pop/urbano.
- **Nuevo** `artist_n_tracks`: tamaño del catálogo del artista principal (metadato público, sin el target). Se evalúa
  como variable en la ablación y se usa para medir el desempeño por tipo de artista (Sección 15).
- **Nuevo** `artist_primary`: primer artista de la lista; define los grupos del split.

In [ ]:
df = pd.read_csv(ubicar('spotify_tracks_clean.csv'))

core_audio = ['explicit', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']
n_antes = len(df)
df = df.dropna(subset=core_audio).reset_index(drop=True)
df['explicit']   = df['explicit'].astype(int)
df['is_hit']     = (df['popularity'] >= UMBRAL_HIT).astype(int)
df['artists']    = df['artists'].fillna('desconocido').astype(str)
df['album_name'] = df['album_name'].fillna('desconocido').astype(str)

genre_cols = [c for c in df.columns if c.startswith('genre_')]
activos = df[genre_cols].sum(axis=1)
df['genre'] = df[genre_cols].idxmax(axis=1).str.replace('genre_', '', regex=False)
df.loc[activos == 0, 'genre'] = 'unknown'

# Artista principal (primer nombre de la lista): define los GRUPOS del split.
# Se usa el principal y no el string completo para que "A" y "A;B" queden del mismo lado.
df['artist_primary'] = df['artists'].str.split(';').str[0].str.strip()

# Ingeniería de características (EV2)
df['duration_min']     = df['duration_ms'] / 60000
df['is_short']         = (df['duration_min'] < 3).astype(int)
df['is_long']          = (df['duration_min'] > 5).astype(int)
df['n_artists']        = df['artists'].str.split(';').str.len()
df['energy_x_dance']   = df['energy'] * df['danceability']
df['valence_x_energy'] = df['valence'] * df['energy']
df['explicit_x_dance'] = df['explicit'] * df['danceability']
# Tamaño del catálogo del artista principal (metadato público, sin información del target).
# Se usa para ANALIZAR el desempeño por tipo de artista; la ablación de EV2 mostró que como variable no aporta.
df['artist_n_tracks']  = df.groupby('artist_primary')['track_id'].transform('size')

audio_features = ['duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode',
                  'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence',
                  'tempo', 'time_signature']
engineered = ['duration_min', 'is_short', 'is_long', 'n_artists',
              'energy_x_dance', 'valence_x_energy', 'explicit_x_dance']
catalogo   = ['artist_n_tracks']
num_cols   = audio_features + engineered            # numéricas "musicales"
FEATURES_LANZAMIENTO = num_cols + genre_cols              # set final: disponible al lanzar (decidido en EV2, Sección 8)
cat_cols   = ['artists', 'album_name']              # solo para la ablación (Target Encoding)
X_cols     = FEATURES_LANZAMIENTO + catalogo + cat_cols   # catálogo y TE solo para ablación/auditoría

assert df['is_hit'].isin([0, 1]).all(), 'Target mal formado'
assert df[FEATURES_LANZAMIENTO].notna().all().all(), 'Nulos en variables del modelo'
print(f"Filas: {n_antes:,} -> {len(df):,} | Géneros one-hot: {len(genre_cols)} | "
      f"Artistas principales: {df['artist_primary'].nunique():,}")
print(f"Prevalencia de hit global: {df['is_hit'].mean():.4f}")
print(f"Canciones repetidas (mismo nombre y artista, distinto track_id): "
      f"{df.duplicated(['track_name', 'artists']).sum():,}")

## 4. Síntesis del análisis exploratorio y de sesgos (EV1)

### Calidad y estructura del dato
- El dataset original tenía 114.000 filas y se redujo a 88.732 canciones únicas por `track_id`, consolidando
  duplicados entre géneros con codificación multi-etiqueta.
- **Hallazgo nuevo:** aun así existen miles de canciones repetidas con **otro `track_id`** (misma canción y
  artista, publicada en un single y en un álbum o compilado). Con un split aleatorio, una copia cae en train y otra
  en test: el modelo "ya conoce" la respuesta. Es una de las razones para separar por artista.
- `popularity` concentra valores bajos; el umbral 50 ≈ percentil 75.
- `duration_ms`, `speechiness` e `instrumentalness` son asimétricas; `loudness` tiene atípicos en la cola
  inferior. Los árboles no requieren transformarlas; la Regresión Logística usa escalamiento estándar.

### Sesgos identificados en EV1 y su impacto en el modelamiento
1. **Muestreo por duplicación** (corregido en EV1 por `track_id`; el residuo por nombre se controla con el split por artista).
2. **Recencia**: `popularity` mide reproducciones recientes; el catálogo histórico queda subrepresentado entre los hits.
3. **Retroalimentación positiva (efecto Mateo)**: lo promocionado acumula reproducciones; el modelo puede aprender
   "quién ya tuvo marketing" en vez de "qué canción tiene potencial".
4. **Homogeneización cultural**: géneros de nicho con tasas de hit muy bajas por menor exposición.
5. **Prevalencia desigual por género**: manifestación operativa de los anteriores (Sección 5).

## 5. Cuantificación de la prevalencia desigual por género

In [ ]:
prevalencia = (df.groupby('genre')['is_hit'].agg(n_total='size', n_hits='sum')
               .assign(hit_rate=lambda d: d['n_hits'] / d['n_total']).reset_index())
print("PEOR A MEJOR (menor tasa de hit primero)")
display(prevalencia.sort_values('hit_rate').head(12).round(4))
print("MEJOR A PEOR (mayor tasa de hit primero)")
display(prevalencia.sort_values('hit_rate', ascending=False).head(12).round(4))

n_bajo = (prevalencia['hit_rate'] < 0.05).sum()
print(f"Géneros: {len(prevalencia)} | con tasa de hit < 5%: {n_bajo} ({n_bajo / len(prevalencia):.1%}) | "
      f"sin ningún hit: {(prevalencia['n_hits'] == 0).sum()}")
print(f"Brecha: el género con más hits tiene una tasa {prevalencia['hit_rate'].max():.2f}, "
      f"el de menos {prevalencia['hit_rate'].min():.2f}.")

## 6. Partición por artista (corrección principal)

### Por qué no un split aleatorio
Con un split aleatorio (el de la versión evaluada), canciones del **mismo artista**, del **mismo álbum** y hasta
la **misma canción** con distinto `track_id` quedan a ambos lados. Variables como la popularidad media del álbum
o del artista se calculan con canciones "hermanas" del test, y el modelo se evalúa sobre artistas que ya conoce.
El resultado (ROC-AUC 0.94, PR-AUC 0.85) era **optimista**.

### Decisión
`GroupShuffleSplit` con grupo = artista principal: **ningún artista aparece en train y test**. Así se mide el caso
real y más difícil del negocio: **predecir el éxito de canciones de artistas que el modelo nunca vio**.
Las métricas bajarán respecto a la versión evaluada; esa caída es la **magnitud de la fuga** que se elimina.

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
idx_tr, idx_te = next(gss.split(df, df['is_hit'], groups=df['artist_primary']))
df_train, df_test = df.iloc[idx_tr].copy(), df.iloc[idx_te].copy()

X_train, X_test = df_train[X_cols].copy(), df_test[X_cols].copy()
y_train, y_test = df_train['is_hit'].copy(), df_test['is_hit'].copy()
g_train, g_test = df_train['genre'].copy(), df_test['genre'].copy()
grupos_train    = df_train['artist_primary'].to_numpy()

# Verificaciones de la partición
solapados = set(df_train['artist_primary']) & set(df_test['artist_primary'])
assert len(solapados) == 0, 'Hay artistas en train y test'
albums_compartidos = df_test['album_name'].isin(set(df_train['album_name'])).mean()
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Train: {len(df_train):,} canciones / {df_train['artist_primary'].nunique():,} artistas")
print(f"Test : {len(df_test):,} canciones / {df_test['artist_primary'].nunique():,} artistas (ninguno visto en train)")
print(f"Hit rate train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")
print(f"Géneros presentes en test: {g_test.nunique()} de {df['genre'].nunique()}")
print(f"Canciones de test cuyo nombre de álbum también aparece en train: {albums_compartidos:.1%} "
      "(compilados como 'Greatest Hits' o 'Mega Hits'; ver ablación)")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

## 7. Pipeline de preprocesamiento

Todo paso que aprende de los datos (Target Encoding en la ablación, imputación, escalamiento) va **dentro de un
`Pipeline`**, de modo que en validación cruzada se reajusta en cada pliegue y nunca usa filas de validación.
- Imputación por mediana (defensiva) y escalamiento estándar: indispensable para Regresión Logística, indiferente
  para árboles; se aplica a todos para que los tres modelos compartan exactamente la misma entrada.
- `te_cols`: columnas codificadas con `TargetEncoder` (`cv=5`, `smooth='auto'`), **solo para la ablación**.

In [ ]:
def construir_pipeline(modelo, cols_num, te_cols=()):
    trans = [('num', 'passthrough', list(cols_num))]
    if te_cols:
        trans.append(('te', TargetEncoder(smooth='auto', cv=5, random_state=RANDOM_STATE), list(te_cols)))
    return Pipeline([
        ('prep', ColumnTransformer(trans, remainder='drop')),
        ('imp',  SimpleImputer(strategy='median')),
        ('esc',  StandardScaler()),
        ('clf',  modelo),
    ])

def xgb_ev2(**kw):
    p = dict(n_estimators=800, learning_rate=0.03, max_depth=6, subsample=0.8, colsample_bytree=0.8,
             scale_pos_weight=scale_pos_weight, eval_metric='aucpr', tree_method='hist',
             random_state=RANDOM_STATE, n_jobs=-1)
    p.update(kw)
    return XGBClassifier(**p)
print("Pipeline definido.")

## 8. Ablación: ¿cuánto aporta cada fuente de información? (IE8)

Se entrena el mismo XGBoost (configuración liviana para acotar el tiempo) agregando bloques de variables uno a uno.
Cada conjunto se evalúa con validación cruzada de 3 pliegues **sobre train** de dos formas:
- **Aleatoria** (`StratifiedKFold`): reproduce la evaluación de la versión anterior.
- **Por artista** (`StratifiedGroupKFold`): artistas de validación nunca vistos.

La **diferencia entre ambas columnas** cuantifica la fuga de información de cada bloque. El test no se usa aquí:
la decisión de qué variables usar no debe tomarse mirando el test.

In [ ]:
VARIANTES = {
    '1. Solo audio':              (audio_features, ()),
    '2. + Ingeniería':            (num_cols, ()),
    '3. + Género':                (num_cols + genre_cols, ()),
    '4. + Catálogo del artista':  (FEATURES_LANZAMIENTO + catalogo, ()),
    '5. + TE artista':            (FEATURES_LANZAMIENTO + catalogo, ('artists',)),
    '6. + TE álbum':              (FEATURES_LANZAMIENTO + catalogo, ('artists', 'album_name')),
}
xgb_liviano = xgb_ev2(n_estimators=300, learning_rate=0.08)
cv_aleatorio = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
cv_grupo     = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def pr_auc_cv(pipe, cv, groups=None):
    vals = []
    for tr, va in cv.split(X_train, y_train, groups):
        m = clone(pipe).fit(X_train.iloc[tr], y_train.iloc[tr])
        vals.append(average_precision_score(y_train.iloc[va], m.predict_proba(X_train.iloc[va])[:, 1]))
    return np.mean(vals)

t0 = time.perf_counter()
filas = []
for nombre, (cols, te) in VARIANTES.items():
    pipe = construir_pipeline(xgb_liviano, cols, te)
    a = pr_auc_cv(pipe, cv_aleatorio)
    g = pr_auc_cv(pipe, cv_grupo, grupos_train)
    filas.append({'Conjunto de variables': nombre, 'Nº variables': len(cols) + len(te),
                  'PR-AUC split aleatorio': a, 'PR-AUC split por artista': g,
                  'Fuga (aleatorio - artista)': a - g})
    print(f"{nombre:28s} aleatorio={a:.3f} | por artista={g:.3f}")
ablacion = pd.DataFrame(filas).set_index('Conjunto de variables')
ablacion['Aporte marginal (por artista)'] = ablacion['PR-AUC split por artista'].diff()
print(f"\nPrevalencia (PR-AUC de un clasificador al azar): {y_train.mean():.3f} | {time.perf_counter() - t0:.0f} s")
display(ablacion.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(ablacion))
ax.bar(x - 0.2, ablacion['PR-AUC split aleatorio'], 0.38, color='#9ec5f4', label='Split aleatorio (versión anterior)')
ax.bar(x + 0.2, ablacion['PR-AUC split por artista'], 0.38, color='#1c5cab', label='Split por artista (corregido)')
ax.axhline(y_train.mean(), color='k', ls='--', lw=1, label='Azar (prevalencia)')
ax.set_xticks(x); ax.set_xticklabels(ablacion.index, rotation=12, fontsize=9)
ax.set_ylabel('PR-AUC (CV=3 en train)'); ax.set_ylim(0, 1)
ax.set_title('Ablación: aporte de cada bloque de variables y fuga por tipo de split'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(DIRS['images'], 'ev2_ablacion.png'), dpi=120); plt.show()

g = ablacion['PR-AUC split por artista']
print("LECTURA AUTOMÁTICA")
print(f"- La música sola (audio) logra PR-AUC {g.iloc[0]:.3f} frente a {y_train.mean():.3f} del azar.")
print(f"- El género agrega {g.iloc[2] - g.iloc[1]:+.3f}: el mercado premia unos géneros más que otros.")
print(f"- El catálogo del artista agrega {g.iloc[3] - g.iloc[2]:+.3f} sin usar el target.")
print(f"- TE de artista con artistas nuevos: {g.iloc[4] - g.iloc[3]:+.3f} "
      "(para un artista nunca visto el encoder solo puede devolver la media global).")
print(f"- TE de álbum: {g.iloc[5] - g.iloc[4]:+.3f} con artistas nuevos y "
      f"{ablacion['Fuga (aleatorio - artista)'].iloc[5]:+.3f} de fuga adicional con split aleatorio.")

### Por qué el TE de álbum "ayuda" incluso con artistas nuevos: es fuga, no señal
Miles de nombres de álbum se repiten entre artistas distintos: son **compilados, playlists y reediciones** ("Greatest
Hits", "Mega Hits", "Daily Pop Mix", "Christmas Songs 2022"). Dentro de un mismo álbum la popularidad es muy homogénea:
por ejemplo, los compilados de temporada tienen tasa de hit 0 porque son copias de canciones cuyo original acumula las
reproducciones. Así, la media del álbum "adivina" la etiqueta de la canción (en ambas direcciones) sin decir nada de
su música, y además solo se conoce **después** del lanzamiento. La celda siguiente lo muestra con datos.

### Decisión sobre las variables del modelo final
Se usa el conjunto **3 (audio + ingeniería + género)** = `FEATURES_LANZAMIENTO`:
- Cumple el criterio de disponibilidad al lanzar (Sección 1).
- El **catálogo del artista** no mejora el PR-AUC con artistas nuevos y premiaría a quien ya tiene más canciones
  publicadas (refuerza el efecto Mateo): se excluye del modelo y se usa solo para auditar (Sección 15).
- El **TE de álbum** se excluye **por principio** (fuga y no disponible), aunque su PR-AUC sea mayor.
- El **TE de artista** se excluye porque con artistas nuevos no aporta (solo devuelve la media) y, como el modelo
  aprende a confiar en él en train, puede empeorar las predicciones para artistas nuevos.

In [ ]:
alb = df.groupby('album_name').agg(n_artistas=('artist_primary', 'nunique'), n=('track_id', 'size'),
                                   hit_rate=('is_hit', 'mean'))
multi = alb[alb['n_artistas'] > 1]
print(f"Álbumes con más de un artista principal: {len(multi):,} ({multi['n'].sum():,} canciones)")
print(f"Tasa de hit en esos álbumes: {df[df['album_name'].isin(multi.index)]['is_hit'].mean():.3f} "
      f"vs {df[~df['album_name'].isin(multi.index)]['is_hit'].mean():.3f} en el resto")
display(multi.sort_values('n_artistas', ascending=False).head(10).round(3))
g3, g4 = ablacion.loc['3. + Género', 'PR-AUC split por artista'], ablacion.loc['4. + Catálogo del artista', 'PR-AUC split por artista']
print(f"Catálogo del artista con artistas nuevos: {g4 - g3:+.3f} de PR-AUC -> "
      + ("no aporta: se excluye (decisión confirmada)." if g4 - g3 < 0.01 else
         "aporta algo, pero se excluye para no premiar el tamaño del catálogo (efecto Mateo); queda como línea futura."))

## 9. Estrategia de balanceo: `class_weight` en lugar de SMOTE

En la versión inicial se aplicó SMOTE y el F1 se estancó en 0.57–0.59. Además, SMOTE sobre las columnas one-hot de
género genera canciones sintéticas con `genre_rock = 0.5` y `genre_pop = 0.5`, que no existen.
Se reemplaza por ponderación de clases, que logra el mismo efecto en la función de pérdida sin datos inválidos:
- Random Forest: `class_weight='balanced_subsample'` · XGBoost: `scale_pos_weight` = negativos/positivos
- Regresión Logística: `class_weight='balanced'`

## 10. Justificación de los tres algoritmos

- **Regresión Logística — baseline lineal interpretable.** Si una frontera lineal ya rinde bien, la relación es
  casi monótona; si no, hay interacciones que justifican modelos más complejos.
- **Random Forest — ensamble de bagging robusto.** Maneja no linealidades, atípicos (`loudness`, `duration_ms`) y
  las 114 columnas de género sin transformación; requiere poco ajuste.
- **XGBoost — ensamble de boosting.** Corrige iterativamente los errores; domina en datos tabulares
  desbalanceados con interacciones; `subsample` y `colsample_bytree` regularizan.

Descartados: **KNN** (distancia degradada en ~140 dimensiones y predicción costosa), **SVM** (costo cuadrático en N),
**redes neuronales** (el techo lo pone el dato, no la arquitectura).

**Criterio de comparación:** mismo pipeline, misma semilla, mismo test. Criterio primario **PR-AUC** (adecuado al
desbalance) medido en **validación**; secundario F2/Recall al umbral óptimo.

In [ ]:
definiciones = {
    'Regresión Logística': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(n_estimators=N_ARBOLES_RF, min_samples_leaf=5, max_features='sqrt',
                                            class_weight='balanced_subsample', random_state=RANDOM_STATE, n_jobs=-1),
    'XGBoost': xgb_ev2(),
}
modelos = {}
for nombre, clf in definiciones.items():
    t0 = time.perf_counter()
    # clone(): cada pipeline recibe su PROPIA copia del modelo (si se comparte el objeto, la Sección 11 lo reentrenaría)
    modelos[nombre] = construir_pipeline(clone(clf), FEATURES_LANZAMIENTO).fit(X_train, y_train)
    print(f"{nombre:20s} entrenado en {time.perf_counter() - t0:5.1f} s")

## 11. Optimización del umbral de decisión (F2) en validación

Con clases desbalanceadas, 0.50 no es un umbral de negocio. Se reserva un **20% del train como validación,
también separado por artista**, se entrena un modelo temporal con el 80% restante y se elige el umbral que
maximiza **F2** (el recall pesa el doble: perder un hit es más caro que una promoción de más).
El mismo conjunto de validación se usa para **elegir el mejor modelo** por PR-AUC; el test sigue aislado.

In [ ]:
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
i_tr, i_val = next(gss_val.split(X_train, y_train, groups=grupos_train))
X_tr, X_val, y_tr, y_val = X_train.iloc[i_tr], X_train.iloc[i_val], y_train.iloc[i_tr], y_train.iloc[i_val]

umbrales, curvas_val, prauc_val = {}, {}, {}
for nombre, clf in definiciones.items():
    m = construir_pipeline(clone(clf), FEATURES_LANZAMIENTO).fit(X_tr, y_tr)   # copia independiente del modelo final
    p = m.predict_proba(X_val)[:, 1]
    prec, rec, thr = precision_recall_curve(y_val, p)
    f2 = 5 * prec * rec / (4 * prec + rec + 1e-9); f1 = 2 * prec * rec / (prec + rec + 1e-9)
    idx = f2[:-1].argmax()
    umbrales[nombre], curvas_val[nombre] = float(thr[idx]), (thr, f1[:-1], f2[:-1])
    prauc_val[nombre] = average_precision_score(y_val, p)
    print(f"[{nombre:20s}] umbral F2 = {thr[idx]:.3f} | F2 = {f2[idx]:.4f} | F1 = {f1[idx]:.4f} | "
          f"PR-AUC validación = {prauc_val[nombre]:.4f}")

MEJOR = max(prauc_val, key=prauc_val.get)
print(f"\nMejor modelo según PR-AUC en VALIDACIÓN: {MEJOR}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (nombre, (thr, f1c, f2c)) in zip(axes, curvas_val.items()):
    ax.plot(thr, f1c, lw=2, color='black', ls='--', label='F1')
    ax.plot(thr, f2c, lw=2, color=COLORES[nombre], label='F2')
    ax.axvline(umbrales[nombre], color='red', ls=':', label=f"óptimo={umbrales[nombre]:.2f}")
    ax.set_title(nombre); ax.set_xlabel('Umbral'); ax.legend(fontsize=8)
axes[0].set_ylabel('Score en validación')
plt.tight_layout(); plt.show()

## 12. Evaluación global en el conjunto de prueba (artistas nunca vistos)

- **F1 / F2**: compromiso precision-recall (F2 alineado al KPI).
- **Recall@Hit**: proporción de hits descubiertos. **Precision**: control del costo de promoción.
- **PR-AUC**: métrica correcta con desbalance; su línea base es la prevalencia (≈0.24).
- **ROC-AUC**: discriminación global. **Brier**: calibración de las probabilidades.

In [ ]:
proba_test = {n: m.predict_proba(X_test)[:, 1] for n, m in modelos.items()}
pred_test  = {n: (proba_test[n] >= umbrales[n]).astype(int) for n in modelos}

def resumen(nombre):
    y_hat, p = pred_test[nombre], proba_test[nombre]
    return {'Modelo': nombre, 'Umbral': round(umbrales[nombre], 3),
            'F1': f1_score(y_test, y_hat), 'F2': fbeta_score(y_test, y_hat, beta=2),
            'Recall': recall_score(y_test, y_hat), 'Precision': precision_score(y_test, y_hat),
            'Accuracy': accuracy_score(y_test, y_hat), 'PR-AUC': average_precision_score(y_test, p),
            'ROC-AUC': roc_auc_score(y_test, p), 'Brier': brier_score_loss(y_test, p)}

resultados = pd.DataFrame([resumen(n) for n in modelos]).set_index('Modelo')
print(f"Prevalencia base en test: {y_test.mean():.4f}")
display(resultados.round(4))
print(f"Modelo elegido en validación: {MEJOR} | ¿es también el mejor PR-AUC en test? "
      f"{'sí' if resultados['PR-AUC'].idxmax() == MEJOR else 'no (' + resultados['PR-AUC'].idxmax() + '), diferencia: ' + format(resultados['PR-AUC'].max() - resultados.loc[MEJOR, 'PR-AUC'], '.4f')}")
print(f"\nReporte detallado — {MEJOR}")
print(classification_report(y_test, pred_test[MEJOR], target_names=['No hit', 'Hit'], zero_division=0))

## 13. Matriz de confusión en términos de negocio
- **VP**: hits identificados · **FN**: hits perdidos (costo de oportunidad) · **FP**: promoción desperdiciada
  (costo presupuestario) · **VN**: canciones correctamente descartadas.

In [ ]:
cm = confusion_matrix(y_test, pred_test[MEJOR])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', cbar=False,
            xticklabels=['Pred: No hit', 'Pred: Hit'], yticklabels=['Real: No hit', 'Real: Hit'])
plt.title(f'Matriz de confusión — {MEJOR} (umbral {umbrales[MEJOR]:.2f})'); plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
waste_rate = fp / (fp + tp)
oportunidad_perdida = fn / (fn + tp)
print(f"VN: {tn:,} | FP: {fp:,} | FN: {fn:,} | VP: {tp:,}")
print(f"Tasa de desperdicio (Waste Rate): {waste_rate:.2%} — de cada 100 canciones promocionadas, "
      f"{waste_rate * 100:.0f} no serían hit.")
print(f"Tasa de oportunidad perdida: {oportunidad_perdida:.2%} de los hits reales no se detectan.")

## 14. Análisis desagregado por género (reporte honesto)
Estados: `ok` (≥10 positivos y ≥10 negativos en test: F1 interpretable), `insuficiente` (1–9 positivos o pocos
negativos: solo PR-AUC como indicio), `sin_positivos` (no hay métricas de la clase hit). Así no se atribuye al
modelo un problema del dato.

In [ ]:
test_eval = pd.DataFrame({'genre': g_test.values, 'y_true': y_test.values,
                          'y_pred': pred_test[MEJOR], 'y_prob': proba_test[MEJOR]})
filas = []
for g_name, grupo in test_eval.groupby('genre'):
    n_pos = int(grupo['y_true'].sum()); n_neg = len(grupo) - n_pos
    fila = {'genre': g_name, 'n_test': len(grupo), 'n_pos': n_pos, 'n_neg': n_neg,
            'hit_rate_test': grupo['y_true'].mean(), 'F1': np.nan, 'Recall': np.nan, 'PR-AUC': np.nan}
    if n_pos >= 10 and n_neg >= 10:
        fila.update(F1=f1_score(grupo['y_true'], grupo['y_pred']), Recall=recall_score(grupo['y_true'], grupo['y_pred']),
                    **{'PR-AUC': average_precision_score(grupo['y_true'], grupo['y_prob'])}, estado='ok')
    elif n_pos > 0 and n_neg > 0:
        fila.update(**{'PR-AUC': average_precision_score(grupo['y_true'], grupo['y_prob'])}, estado='insuficiente')
    else:
        fila['estado'] = 'sin_positivos' if n_pos == 0 else 'insuficiente'
    filas.append(fila)
reporte_genero = pd.DataFrame(filas).sort_values('n_pos', ascending=False).reset_index(drop=True)
print("Diez géneros con más positivos en test"); display(reporte_genero.head(10).round(3))
print("Diez géneros con menos positivos en test"); display(reporte_genero.tail(10).round(3))
print(reporte_genero['estado'].value_counts().to_string())
ok = reporte_genero[reporte_genero['estado'] == 'ok']
print(f"\nF1 en géneros evaluables: media {ok['F1'].mean():.3f} | mínimo {ok['F1'].min():.3f} ({ok.loc[ok['F1'].idxmin(), 'genre']}) "
      f"| máximo {ok['F1'].max():.3f} ({ok.loc[ok['F1'].idxmax(), 'genre']})")

## 15. ¿Se subestima a los artistas emergentes? (medición, IE8)
En la versión anterior se afirmó sin cifras. Aquí se compara el desempeño según el **tamaño del catálogo del
artista** en el dataset: 1 canción (emergente o esporádico), 2–10, y más de 10 (establecido). El catálogo no es una
variable del modelo, así que la tabla mide si el modelo trata distinto a cada tipo de artista **sin saber** cuál es.
La conclusión se genera desde los datos, sin suponer la dirección de antemano.

In [ ]:
seg = pd.cut(df_test['artist_n_tracks'], bins=[0, 1, 10, np.inf], labels=['1 canción', '2–10 canciones', '>10 canciones'])
filas = []
for s in seg.cat.categories:
    m = (seg == s).to_numpy(); yt = y_test.to_numpy()[m]; yp = pred_test[MEJOR][m]; pp = proba_test[MEJOR][m]
    filas.append({'Catálogo del artista': s, 'Canciones': int(m.sum()), 'Hits reales': int(yt.sum()),
                  'Tasa de hit real': yt.mean(), 'Prob. media predicha': pp.mean(),
                  'Recall': recall_score(yt, yp, zero_division=0), 'Precision': precision_score(yt, yp, zero_division=0),
                  'PR-AUC': average_precision_score(yt, pp) if 0 < yt.sum() < len(yt) else np.nan})
artistas = pd.DataFrame(filas).set_index('Catálogo del artista')
display(artistas.round(3))
r1, r10 = artistas.loc['1 canción', 'Recall'], artistas.loc['>10 canciones', 'Recall']
print(f"Recall artistas con 1 canción: {r1:.3f} vs artistas con >10: {r10:.3f} (diferencia {r1 - r10:+.3f}).")
if r1 < r10 - 0.03:
    print("=> Se confirma la hipótesis: el modelo detecta menos hits de artistas emergentes.")
elif r1 > r10 + 0.03:
    print("=> La hipótesis NO se confirma: en este dataset los artistas con 1 canción tienen MÁS hits "
          f"({artistas.loc['1 canción', 'Tasa de hit real']:.2f} vs {artistas.loc['>10 canciones', 'Tasa de hit real']:.2f}) "
          "y el modelo los detecta mejor. Los catálogos grandes son sobre todo clásicos, ambient y artistas históricos.")
else:
    print("=> Diferencia pequeña: no hay evidencia de que el modelo subestime a artistas emergentes.")
print("Calibración: las probabilidades medias predichas superan la tasa real en todos los grupos porque class_weight "
      "infla la clase hit; no deben leerse como probabilidades reales sin calibrar.")

## 16. Aprendizaje no supervisado: PCA y K-Means

- PCA sobre las 14 variables acústicas estandarizadas (escalador ajustado solo con train), conservando ≥85% de varianza.
- Número de clusters por método del codo y silhouette.

In [ ]:
esc_audio = StandardScaler().fit(df_train[audio_features])
A_train = esc_audio.transform(df_train[audio_features]); A_test = esc_audio.transform(df_test[audio_features])
pca = PCA(n_components=0.85, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(A_train)
print(f"Componentes retenidos: {pca.n_components_} | varianza explicada: {pca.explained_variance_ratio_.sum():.3f}")

inercias, silhouettes = [], []
muestra = np.random.default_rng(RANDOM_STATE).choice(len(X_pca), size=min(15000, len(X_pca)), replace=False)
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X_pca)
    inercias.append(km.inertia_); silhouettes.append(silhouette_score(X_pca[muestra], km.labels_[muestra]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(range(2, 8), inercias, marker='o'); axes[0].set_title('Método del codo'); axes[0].set_xlabel('k')
axes[1].plot(range(2, 8), silhouettes, marker='o', color='green'); axes[1].set_title('Silhouette'); axes[1].set_xlabel('k')
plt.tight_layout(); plt.show()
print("Silhouette por k:", {k: round(s, 3) for k, s in zip(range(2, 8), silhouettes)})

### Selección del número de clusters
El silhouette suele ser máximo en k=2–3, pero esos k producen un cluster que absorbe más de la mitad del catálogo.
Se adopta **K=4 como compromiso explícito entre separación estadística e interpretabilidad**, reconociendo que su
silhouette es inferior al óptimo: es una decisión de diseño, no el resultado del criterio estadístico.

In [ ]:
K_FINAL = 4
kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=20).fit(X_pca)
cl_train = kmeans.labels_; cl_test = kmeans.predict(pca.transform(A_test))

perfil = df_train[audio_features].assign(cluster=cl_train, is_hit=y_train.values).groupby('cluster').mean()
perfil['n'] = np.bincount(cl_train)
perfil['lift'] = perfil['is_hit'] / y_train.mean()
display(perfil[['n', 'danceability', 'energy', 'acousticness', 'instrumentalness', 'valence', 'tempo',
                'is_hit', 'lift']].round(3))

for c in range(K_FINAL):
    top = g_train[cl_train == c].value_counts(normalize=True).head(6)
    print(f"Cluster {c}: " + ", ".join(f"{g} ({v:.0%})" for g, v in top.items()))

rec_cluster = []
for c in range(K_FINAL):
    m = (cl_test == c) & (y_test.to_numpy() == 1)
    rec_cluster.append({'cluster': c, 'hits en test': int(m.sum()), f'Recall {MEJOR}': (pred_test[MEJOR][m] == 1).mean()})
display(pd.DataFrame(rec_cluster).set_index('cluster').round(3))

plt.figure(figsize=(9, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cl_train, cmap='viridis', alpha=0.3, s=6)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], s=220, c='red', marker='X', edgecolor='black')
for c, (x0, y0) in enumerate(kmeans.cluster_centers_[:, :2]):
    plt.annotate(str(c), (x0, y0), xytext=(10, 10), textcoords='offset points', fontsize=13, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='black'))
plt.title('Segmentación sonora (PCA + K-Means, K=4)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})'); plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.tight_layout(); plt.show()

### Interpretación (con cautela)
Los clusters son **segmentos sonoros, no géneros**: el K-Means se ajustó solo con audio. Las diferencias de tasa de
hit entre clusters son **moderadas** (en EV2 iban de 0.21 a 0.30) y los géneros dentro de cada cluster son
heterogéneos. Por eso **no** se concluye que "los clusters validan que el modelo capta estructura sonora real":
lo que muestran es que el perfil sonoro por sí solo separa poco el éxito, consistente con la ablación (el audio
solo logra un PR-AUC limitado). Su utilidad está en **auditar** si el modelo descarta sistemáticamente algún perfil
sonoro (tabla de recall por cluster) y en diseñar playlists temáticas.

## 17. Modelo de regresión de popularidad (KPIs MAE, RMSE, R²)

La tabla de KPIs incluye métricas de predicción de valor, por lo que se entrena un `XGBRegressor` sobre
`popularity` (0–100) con el mismo split por artista. Se reportan dos variantes:
- **Solo música** (audio + ingeniería): mide el KPI R² tal como está definido ("porcentaje de la popularidad que se debe solo a la música").
- **Set de lanzamiento**: el mismo conjunto de variables del clasificador.

In [ ]:
pop_train, pop_test = df_train['popularity'], df_test['popularity']
REG_PARAMS = dict(n_estimators=600, learning_rate=0.05, max_depth=7, subsample=0.8, colsample_bytree=0.8,
                  tree_method='hist', random_state=RANDOM_STATE, n_jobs=-1)
regresion, pred_pop = [], {}
for nombre, cols in [('Solo música', num_cols), ('Set de lanzamiento', FEATURES_LANZAMIENTO)]:
    reg = construir_pipeline(XGBRegressor(**REG_PARAMS), cols).fit(X_train, pop_train)
    pred_pop[nombre] = np.clip(reg.predict(X_test), 0, 100)
    regresion.append({'Variante': nombre, 'MAE': mean_absolute_error(pop_test, pred_pop[nombre]),
                      'RMSE': np.sqrt(mean_squared_error(pop_test, pred_pop[nombre])),
                      'R²': r2_score(pop_test, pred_pop[nombre])})
regresion = pd.DataFrame(regresion).set_index('Variante')
baseline_mae = mean_absolute_error(pop_test, np.full(len(pop_test), pop_train.median()))
display(regresion.round(3))
print(f"Referencia: predecir siempre la mediana da MAE = {baseline_mae:.2f}")

## 18. Calidad de la lista promocionada: NDCG@20 y diversidad
- **NDCG@20**: dentro de cada género (una "playlist candidata") se ordenan las canciones por probabilidad y se
  compara con el orden real de popularidad; 1.0 = orden perfecto.
- **Diversity Score**: géneros distintos en el **top-200** de canciones que se promocionarían.

In [ ]:
def ndcg_por_genero(proba, k=20, min_n=40):
    vals = []
    for g in np.unique(g_test):
        m = g_test.to_numpy() == g
        if m.sum() >= min_n:
            vals.append(ndcg_score(pop_test.to_numpy()[m].reshape(1, -1), proba[m].reshape(1, -1), k=k))
    return float(np.mean(vals))

ndcg20 = ndcg_por_genero(proba_test[MEJOR])
top200 = np.argsort(-proba_test[MEJOR])[:200]
diversidad = int(g_test.iloc[top200].nunique())
print(f"NDCG@20 (media por género): {ndcg20:.3f}")
print(f"Géneros distintos en el top-200: {diversidad} | género más frecuente: "
      f"{g_test.iloc[top200].value_counts().index[0]} ({g_test.iloc[top200].value_counts(normalize=True).iloc[0]:.0%})")
print(f"Géneros distintos entre todas las canciones predichas como hit: {g_test[pred_test[MEJOR] == 1].nunique()}")

## 19. Interpretación de sesgos, ética y uso responsable

### Sesgos con evidencia cuantitativa

In [ ]:
g_ = ablacion['PR-AUC split por artista']; a_ = ablacion['PR-AUC split aleatorio']
lineas = [
    f"FUGA DE INFORMACIÓN: con split aleatorio y TE de álbum el PR-AUC en CV es {a_.iloc[5]:.3f}; con artistas "
    f"nuevos y variables disponibles al lanzar es {g_.iloc[2]:.3f}. La diferencia ({a_.iloc[5] - g_.iloc[2]:.3f}) era "
    "optimismo, no capacidad predictiva.",
    f"EFECTO MATEO: la música sola explica R² = {regresion.loc['Solo música', 'R²']:.2f} de la popularidad; el género "
    f"agrega PR-AUC {g_.iloc[2] - g_.iloc[1]:+.3f}. El éxito depende más del mercado (qué géneros premia la plataforma) "
    "que del sonido.",
    f"ARTISTAS EMERGENTES: recall {artistas.loc['1 canción', 'Recall']:.2f} para artistas con 1 canción vs "
    f"{artistas.loc['>10 canciones', 'Recall']:.2f} para artistas con más de 10 "
    + ("(el modelo NO los subestima en este dataset)." if artistas.loc['1 canción', 'Recall'] >= artistas.loc['>10 canciones', 'Recall'] - 0.03
       else "(el modelo los subestima)."),
    f"HOMOGENEIZACIÓN: {(prevalencia['hit_rate'] < 0.05).sum()} géneros con tasa de hit < 5%; el top-200 incluye "
    f"{diversidad} géneros (el ranking se concentra en los géneros con más hits).",
    "CALIBRACIÓN: class_weight sube las probabilidades de la clase hit; sirven para ordenar, no como probabilidad real.",
]
for l in lineas:
    print("- " + "\n  ".join(textwrap.wrap(l, 105)))

### Datos personales y marco normativo
El dataset no contiene datos personales de oyentes; nombres de artistas, canciones y álbumes son públicos. La
popularidad deriva de patrones de consumo agregados, regulados por el RGPD y la Ley 19.628 en Chile. El uso de un
modelo así para decidir presupuesto que afecta ingresos de artistas exige transparencia (AI Act europeo).

### Riesgos éticos
1. **Invisibilización de artistas emergentes**: si el modelo se usa como filtro, el círculo de retroalimentación se cierra.
2. **Homogeneización sonora**: si todos los sellos usan modelos similares, el catálogo tiende a la uniformidad.
3. **Opacidad**: las decisiones sobre canciones concretas no son auditables por los artistas.
4. **Manipulación**: *stream farming* puede contaminar la variable objetivo en reentrenamientos.

### Pregunta de la presentación: ¿cómo dar más peso a artistas que invierten en publicidad?
Cuatro alternativas técnicas:
1. **`sample_weight` al entrenar**, con más peso para canciones de esos artistas:
   `pipe.fit(X_train, y_train, clf__sample_weight=pesos)`.
2. **Variable de campaña o presupuesto invertido** como feature (requiere datos de marketing del sello).
3. **Umbral distinto para ese segmento** (más bajo para artistas con inversión comprometida).
4. **Reordenar la lista final con un puntaje de negocio**: `puntaje = probabilidad × valor esperado del artista`.

**Riesgo:** todas amplifican el **efecto Mateo** descrito en la Sección 4: quien ya invierte recibe más promoción y
acumula más popularidad, lo que en el siguiente reentrenamiento refuerza el sesgo. Si se aplican, deben ir
acompañadas de una **cuota protegida para artistas sin historial** y de monitoreo del recall por tipo de artista
(Sección 15). La opción 4 es la más segura porque no altera el modelo, solo la decisión, y es reversible.

### Recomendaciones de uso responsable
1. Usar el modelo como **señal complementaria**, no como filtro excluyente; decisión final con criterio humano.
2. Reservar ≈20% del presupuesto para artistas sin historial.
3. Reportar en cada iteración el desempeño por género y por tamaño de catálogo.
4. Documentar el umbral (F2 = prioriza recall sobre precisión) como política explícita.
5. Recalibrar periódicamente con datos que incluyan las intervenciones promocionales.

## 20. Comparación con versiones previas y conclusiones

Las versiones anteriores se midieron con un **split aleatorio** (artistas compartidos entre train y test), por lo que
sus cifras no son directamente comparables: la caída de la versión corregida es, en gran parte, la fuga eliminada.

In [ ]:
comparativa = pd.DataFrame([
    {'Versión': 'Inicial (SMOTE + umbral 0.5, split aleatorio)', 'F1': 0.5603, 'Recall': 0.5835, 'PR-AUC': np.nan, 'ROC-AUC': np.nan},
    {'Versión': 'EV2 evaluada (TE artista/álbum, split aleatorio)', 'F1': 0.7246, 'Recall': 0.8962, 'PR-AUC': 0.8487, 'ROC-AUC': 0.9389},
    {'Versión': f'EV2 corregida ({MEJOR}, split por artista, sin TE)', 'F1': resultados.loc[MEJOR, 'F1'],
     'Recall': resultados.loc[MEJOR, 'Recall'], 'PR-AUC': resultados.loc[MEJOR, 'PR-AUC'], 'ROC-AUC': resultados.loc[MEJOR, 'ROC-AUC']},
]).set_index('Versión')
display(comparativa.round(4))

In [ ]:
conclusiones = [
    f"1. TECHO PREDICTIVO REAL. Con artistas nunca vistos y solo información disponible al lanzar, {MEJOR} logra "
    f"PR-AUC {resultados.loc[MEJOR, 'PR-AUC']:.3f} (azar = {y_test.mean():.3f}) y ROC-AUC {resultados.loc[MEJOR, 'ROC-AUC']:.3f}. "
    f"Es menor que el 0.849 reportado antes porque esa cifra incluía fuga del álbum y del artista (Sección 8).",
    f"2. LA MÚSICA EXPLICA POCO. Solo audio: PR-AUC {ablacion['PR-AUC split por artista'].iloc[0]:.3f}; solo música "
    f"(audio + ingeniería): R² {regresion.loc['Solo música', 'R²']:.2f}. El género aporta la mayor parte de la señal útil; el catálogo del "
    "artista no aporta con artistas nuevos.",
    f"3. BALANCEO. class_weight se eligió para no crear filas inválidas en el one-hot de género; la comparación con "
    "SMOTE mezcla varios cambios, por lo que no aísla su efecto.",
    f"4. UMBRAL. El umbral óptimo por F2 ({umbrales[MEJOR]:.2f}) es menor que 0.5: con desbalance, elegir el umbral es "
    f"parte del pipeline. Costo: Waste Rate {waste_rate:.1%}.",
    f"5. ARTISTAS EMERGENTES. Recall {artistas.loc['1 canción', 'Recall']:.2f} en artistas con 1 canción vs "
    f"{artistas.loc['>10 canciones', 'Recall']:.2f} con más de 10: "
    + ("la hipótesis de la versión anterior no se confirma; el sesgo relevante es por GÉNERO, no por catálogo."
       if artistas.loc['1 canción', 'Recall'] >= artistas.loc['>10 canciones', 'Recall'] - 0.03
       else "el modelo favorece a artistas con catálogo."),
    "6. NO SUPERVISADO. Los 4 segmentos sonoros tienen tasas de hit parecidas; sirven para auditar sesgos por perfil "
    "sonoro, no como prueba de que el modelo capte 'calidad musical'.",
]
for c in conclusiones:
    print("\n".join(textwrap.wrap(c, 105)), end="\n\n")

### Limitaciones
- Faltan variables de contexto: seguidores, sello, playlists editoriales, fecha de lanzamiento.
- `popularity` es una métrica de plataforma con sesgo de recencia; no mide calidad.
- El modelo no puede descontar intervenciones promocionales previas (hereda el efecto Mateo).
- Géneros de nicho con 0–3 positivos en test no permiten métricas individuales estables.
- `artist_n_tracks` se cuenta dentro de este dataset (muestra de ~1.000 canciones por género), no en todo Spotify.
- Las probabilidades no están calibradas (class_weight las desplaza hacia la clase hit).

### Líneas futuras
Optimización de hiperparámetros, ensambles y validación cruzada por artista (desarrollados en **EV3**);
embeddings de audio; datos de campañas para separar el efecto marketing del efecto música.

## 21. Verificación de KPIs (la misma tabla definida en la Sección 1)
Los mensajes se generan con la **misma condición** que se evalúa, para que la salida no pueda contradecir al código.

In [ ]:
r = resultados.loc[MEJOR]
evaluacion = {   # KPI: (valor, se cumple)
    'MAE':             (regresion.loc['Set de lanzamiento', 'MAE'],  regresion.loc['Set de lanzamiento', 'MAE'] <= 10),
    'RMSE':            (regresion.loc['Set de lanzamiento', 'RMSE'], regresion.loc['Set de lanzamiento', 'RMSE'] < 12),
    'R2':              (regresion.loc['Solo música', 'R²'],           regresion.loc['Solo música', 'R²'] >= 0.45),
    'Recall@Hit':      (r['Recall'],  r['Recall'] >= 0.70),
    'PR-AUC':          (r['PR-AUC'],  r['PR-AUC'] > 0.50),
    'F2-Score':        (r['F2'],      r['F2'] > 0.65),
    'NDCG@20':         (ndcg20,       ndcg20 > 0.80),
    'Waste Rate':      (waste_rate,   waste_rate <= 0.15),
    'Diversity Score': (diversidad,   diversidad >= 15),
}
kpi_final = kpis[['KPI', 'Tipo de Analisis', 'Meta Sugerida']].copy()
kpi_final['Valor'] = kpi_final['KPI'].map(lambda k: round(float(evaluacion[k][0]), 4))
kpi_final['Estado'] = kpi_final['KPI'].map(lambda k: 'CUMPLE' if evaluacion[k][1] else 'NO CUMPLE')
print(f"Modelo: {MEJOR} | umbral {umbrales[MEJOR]:.2f} | test con artistas nunca vistos")
display(kpi_final)
print(f"KPIs cumplidos: {(kpi_final['Estado'] == 'CUMPLE').sum()}/{len(kpi_final)}")

if evaluacion['Waste Rate'][1]:
    print(f"Waste Rate {waste_rate:.1%} dentro de la meta del 15%.")
else:
    print(f"Waste Rate {waste_rate:.1%} SUPERA la meta del 15%: es consecuencia de optimizar F2 (prioriza recall). "
          "Si el negocio exige ≤15%, hay que subir el umbral y aceptar mucho menos recall; EV3 cuantifica ese "
          "trade-off y propone redefinir la meta respecto del azar (promocionar al azar desperdicia "
          f"{1 - y_test.mean():.0%}).")
if not evaluacion['R2'][1]:
    print(f"R² solo audio = {regresion.loc['Solo música', 'R²']:.2f} < 0.45: la música sola no explica la popularidad "
          "(hallazgo sobre el negocio, consistente con el efecto Mateo).")

## 22. Exportación de resultados
Se guarda el test enriquecido con las predicciones del mejor modelo, el cluster sonoro y las métricas por género,
en `data/` (para el informe) y las tablas de esta evaluación en `reports/`.

In [ ]:
salida = df_test[['track_id', 'track_name', 'artists', 'artist_primary', 'artist_n_tracks', 'album_name', 'genre',
                  'popularity', 'is_hit']].copy()
for n in modelos:
    salida[f'prob_{n.split()[0].lower()}'] = proba_test[n]
salida['pred_hit'] = pred_test[MEJOR]
salida['popularidad_estimada'] = pred_pop['Set de lanzamiento']
salida['cluster_sonoro'] = cl_test
salida = salida.merge(reporte_genero[['genre', 'F1', 'PR-AUC', 'n_pos', 'estado']].rename(columns={
    'F1': 'genre_f1_test', 'PR-AUC': 'genre_pr_auc_test', 'n_pos': 'genre_n_pos_test', 'estado': 'genre_estado_test'}),
    on='genre', how='left')
salida.to_csv(os.path.join(DIRS['data'], 'spotify_tracks_evaluated.csv'), index=False)
for nombre, t in {'ev2_ablacion.csv': ablacion, 'ev2_resultados_test.csv': resultados, 'ev2_artistas.csv': artistas,
                  'ev2_regresion.csv': regresion, 'ev2_kpis.csv': kpi_final}.items():
    t.to_csv(os.path.join(DIRS['reports'], nombre))
print(f"Exportado: {salida.shape} -> {os.path.join(DIRS['data'], 'spotify_tracks_evaluated.csv')}")